In [8]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
from sklearn.metrics import classification_report

DATASET_PATH = r"D:\waste_project\waste-classification-system-II\dataset\New Trash Classfication Dataset\new-dataset-trash-type-v2"

IMG_SIZE = (96, 96)
BATCH_SIZE = 32

CLASS_NAMES = ["organic", "cardboard", "paper", "plastic", "textile"]
NUM_CLASSES = len(CLASS_NAMES)


In [17]:
base_model = keras.applications.MobileNetV3Small(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
    pooling="avg"
)

base_model.trainable = False  # freeze pretrained backbone

model = keras.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),

    # FIX: wrap preprocess_input inside a Lambda layer
    layers.Lambda(keras.applications.mobilenet_v3.preprocess_input),

    base_model,

    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                      │ (None, 96, 96, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ MobileNetV3Small (Functional)        │ (None, 576)                 │         939,120 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_12 (Dropout)                 │ (None, 576)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_22 (Dense)                     │ (None, 128)                 │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_13 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_23 (Dense)                     │ (None, 5)                   │             645 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,013,621 (3.87 MB)

 Trainable params: 74,501 (291.02 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [14]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [15]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=6,
            restore_best_weights=True
        )
    ]
)


Epoch 1/25
115/115 ━━━━━━━━━━━━━━━━━━━━ 13s 44ms/step - accuracy: 0.2146 - loss: 1.6085 - val_accuracy: 0.1908 - val_loss: 1.6122
Epoch 2/25
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - accuracy: 0.2222 - loss: 1.5861 - val_accuracy: 0.3457 - val_loss: 1.5206
Epoch 3/25
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 46ms/step - accuracy: 0.3118 - loss: 1.4998 - val_accuracy: 0.3642 - val_loss: 1.4282
Epoch 4/25
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 45ms/step - accuracy: 0.3331 - loss: 1.4539 - val_accuracy: 0.3904 - val_loss: 1.4051
Epoch 5/25
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 46ms/step - accuracy: 0.3407 - loss: 1.4326 - val_accuracy: 0.3926 - val_loss: 1.3906
Epoch 6/25
115/115 ━━━━━━━━━━━━━━━━━━━━ 8s 47ms/step - accuracy: 0.3426 - loss: 1.4256 - val_accuracy: 0.3850 - val_loss: 1.3907
Epoch 7/25
115/115 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.3540 - loss: 1.4187 - val_accuracy: 0.3969 - val_loss: 1.3802
Epoch 8/25
115/115 ━━━━━━━━━━━━━━━━━━━━ 8s 54ms/step - accuracy: 0.3614 - loss: 1.4183 - val_acc

KeyboardInterrupt: 

In [12]:
base_model = keras.applications.MobileNetV3Large(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
    pooling="avg"
)

base_model.trainable = False  # freeze backbone

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = keras.applications.mobilenet_v3.preprocess_input(inputs)

x = base_model(x, training=False)
x = layers.Dropout(0.35)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()


C:\Users\Asghar\anaconda3\envs\waste_env\lib\site-packages\keras\src\applications\mobilenet_v3.py:519: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)           │ (None, 96, 96, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ MobileNetV3Large (Functional)        │ (None, 960)                 │       2,996,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 960)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_16 (Dense)                     │ (None, 128)                 │         123,008 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_17 (Dense)                     │ (None, 5)                   │             645 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,120,005 (11.90 MB)

 Trainable params: 123,653 (483.02 KB)

 Non-trainable params: 2,996,352 (11.43 MB)

In [6]:
EPOCHS = 20

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=8,
        restore_best_weights=True,
        mode="max",
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy",
        factor=0.5,
        patience=4,
        min_lr=1e-5,
        mode="max",
        verbose=1,
    ),
]


In [7]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight,
)


Epoch 1/20
115/115 ━━━━━━━━━━━━━━━━━━━━ 43s 230ms/step - accuracy: 0.2100 - loss: 1.6105 - val_accuracy: 0.1919 - val_loss: 1.6101 - learning_rate: 0.0010
Epoch 2/20
115/115 ━━━━━━━━━━━━━━━━━━━━ 24s 194ms/step - accuracy: 0.1857 - loss: 1.6104 - val_accuracy: 0.2148 - val_loss: 1.6094 - learning_rate: 0.0010
Epoch 3/20
115/115 ━━━━━━━━━━━━━━━━━━━━ 24s 191ms/step - accuracy: 0.1852 - loss: 1.6102 - val_accuracy: 0.1919 - val_loss: 1.6101 - learning_rate: 0.0010
Epoch 4/20
115/115 ━━━━━━━━━━━━━━━━━━━━ 24s 194ms/step - accuracy: 0.1833 - loss: 1.6101 - val_accuracy: 0.1919 - val_loss: 1.6103 - learning_rate: 0.0010
Epoch 5/20
 98/115 ━━━━━━━━━━━━━━━━━━━━ 3s 224ms/step - accuracy: 0.2068 - loss: 1.6050

KeyboardInterrupt: 

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f"\nFinal val accuracy: {acc:.2%}")

y_true, y_pred = [], []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


In [ ]:
def representative_data_gen():
    for images, _ in train_ds.take(50):
        yield [tf.cast(images, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

with open("waste_5class_tiny_mbnetv2_int8.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model size (bytes):", len(tflite_model))
